# ChuckleNet: Streaming WavLM + Fusion Training

Based on `train_wavlm_audio_v2.py` — proven working design:
- ONE audio file in memory at a time (avoids Colab OOM)
- Process all utterances from that file → save embeddings → free memory
- 3-phase training: A=frozen WavLM, B=partial unfreeze, C=joint

**Runtime**: ~3 hours on T4 GPU
**RAM**: < 4GB (one audio file + model + batch at a time)

In [ ]:
#@title 1. Setup
!pip install transformers torch torchaudio librosa scikit-learn -q
!nvidia-smi

In [ ]:
#@title 2. Mount & Upload
from google.colab import drive, files
drive.mount('/content/drive')
import os, json

# Upload utterances
UTT_FILE = '/content/utt_data'
if not os.path.exists(UTT_FILE):
    print('📤 Upload aligned_utterances.jsonl (~5MB)')
    uploaded = files.upload()
    for fn in uploaded.keys():
        os.rename(fn, UTT_FILE)

with open(UTT_FILE) as f:
    utterances = [json.loads(l) for l in f if l.strip()]
print(f'{len(utterances)} utterances loaded')
pos = sum(1 for u in utterances if u.get('label_any',0)==1)
print(f'Positive: {pos} ({100*pos/len(utterances):.1f}%)')

In [ ]:
#@title 3. Audio Map
import glob

AUDIO_BASE = '/content/drive/MyDrive/laughter_prediction_backup/audio'
audio_map = {}
for ext in ['*.mp3', '*.wav', '*.m4a']:
    for p in glob.glob(f'{AUDIO_BASE}/**/{ext}', recursive=True):
        vid = os.path.basename(p).rsplit('.',1)[0]
        audio_map[vid] = p

covered = sum(1 for u in utterances if u['video_id'] in audio_map)
print(f'Audio files: {len(audio_map)}, Coverage: {covered}/{len(utterances)}')

In [ ]:
#@title 4. Stream Extract: One Video at a Time → Save Embeddings
import torch, librosa, gc, os
from collections import defaultdict
from tqdm.auto import tqdm

device = torch.device('cuda')
from transformers import WavLMModel
wavlm = WavLMModel.from_pretrained('microsoft/wavlm-base-plus').to(device).eval()
SR = 16000
PAD = 0.2
OUTPUT_DIR = '/content/embeddings'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Group utterances by video
video_utts = defaultdict(list)
for u in utterances:
    if u['video_id'] in audio_map:
        video_utts[u['video_id']].append(u)

videos = sorted(video_utts.keys())
print(f'Videos to process: {len(videos)}')

# Check resume point
done_files = sorted([f for f in os.listdir(OUTPUT_DIR) if f.endswith('.pt')])
start_from = len(done_files)
if start_from > 0:
    print(f'Reuming from video {start_from}/{len(videos)}')

for vi, vid in enumerate(tqdm(videos[start_from:], desc='Videos')):
    utts = video_utts[vid]
    
    # Load ONE audio file at a time
    try:
        y, sr = librosa.load(audio_map[vid], sr=SR, mono=True)
    except:
        torch.save({'ids': [u['utterance_id'] for u in utts],
                    'embs': torch.zeros(len(utts), 768)},
                    f'{OUTPUT_DIR}/{vid}.pt')
        continue
    
    embs, ids = [], []
    
    for u in utts:
        t0 = max(0, u['start'] - PAD)
        t1 = min(len(y)/SR, u['end'] + PAD)
        clip = y[int(t0*SR):int(t1*SR)]
        
        if len(clip) < int(0.1*SR):
            clip = torch.zeros(int(0.1*SR))
        else:
            clip = torch.from_numpy(clip.astype('float32'))
        
        with torch.no_grad():
            out = wavlm(clip.unsqueeze(0).to(device))
            emb = out.last_hidden_state.mean(1).squeeze(0).float().cpu()
        
        embs.append(emb)
        ids.append(u['utterance_id'])
    
    # Save video embeddings immediately
    torch.save({
        'ids': ids,
        'embs': torch.stack(embs)
    }, f'{OUTPUT_DIR}/{vid}.pt')
    
    # FREE MEMORY immediately
    del y, embs, clip, out, emb
    torch.cuda.empty_cache()
    gc.collect()

print(f'\n✅ All {len(videos)} videos extracted!')

In [ ]:
#@title 5. Merge All Embeddings → Save
import glob, torch

OUTPUT_DIR = '/content/embeddings'
all_ids, all_embs = [], []

for vf in sorted(glob.glob(f'{OUTPUT_DIR}/*.pt')):
    d = torch.load(vf, map_location='cpu')
    all_ids.extend(d['ids'])
    all_embs.append(d['embs'])
    print(f'{os.path.basename(vf)}: {d["embs"].shape[0]} embeddings')

final = torch.cat(all_embs, dim=0)
print(f'Merged: {final.shape}')

OUTPUT = '/content/drive/MyDrive/wavlm_embeddings_15k'
torch.save({
    'embeddings': final,
    'utterance_ids': all_ids,
    'model': 'microsoft/wavlm-base-plus',
    'pooling': 'mean'
}, OUTPUT)
print(f'Saved to {OUTPUT}')
print(f'Size: {os.path.getsize(OUTPUT)/1024**2:.1f} MB')

In [ ]:
#@title 6. Phase A: Frozen WavLM + Classifier (Baseline)
import torch.nn as nn
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import f1_score
import random, gc

DEVICE = torch.device('cuda')

# Load embeddings
emb_data = torch.load('/content/drive/MyDrive/wavlm_embeddings_15k', map_location='cpu')
emb_dict = {uid: emb for uid, emb in zip(emb_data['utterance_ids'], emb_data['embeddings'])}
print(f'Embeddings: {len(emb_dict)}')

# Build dataset (text + audio embeddings)
tokenizer = AutoTokenizer.from_pretrained('FacebookAI/xlm-roberta-base')

class AudioTextDataset(Dataset):
    def __init__(self, utts, emb_dict, tokenizer):
        self.utts = [u for u in utts if u['utterance_id'] in emb_dict]
        self.tokenizer = tokenizer
        self.emb = emb_dict
    def __len__(self): return len(self.utts)
    def __getitem__(self, i):
        u = self.utts[i]
        enc = self.tokenizer(u['text'], max_length=128, padding='max_length', truncation=True, return_tensors='pt')
        return {
            'input_ids': enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'audio_emb': self.emb[u['utterance_id']].float(),
            'label': torch.tensor(u.get('label_any',0), dtype=torch.long)
        }

def collate(batch):
    return {
        'input_ids': torch.stack([b['input_ids'] for b in batch]),
        'attention_mask': torch.stack([b['attention_mask'] for b in batch]),
        'audio_emb': torch.stack([b['audio_emb'] for b in batch]),
        'labels': torch.stack([b['label'] for b in batch])
    }

# Video-level split
from collections import defaultdict
vid_items = defaultdict(list)
for i, u in enumerate(utterances):
    if u['utterance_id'] in emb_dict:
        vid_items[u['video_id']].append(i)

random.seed(42)
vids = sorted(vid_items.keys())
random.shuffle(vids)
n_val = max(1, len(vids)//10)
val_vids = set(vids[:n_val])

train_utts = [u for u in utterances if u['utterance_id'] in emb_dict and u['video_id'] not in val_vids]
val_utts = [u for u in utterances if u['utterance_id'] in emb_dict and u['video_id'] in val_vids]

train_ds = AudioTextDataset(train_utts, emb_dict, tokenizer)
val_ds = AudioTextDataset(val_utts, emb_dict, tokenizer)
print(f'Train: {len(train_ds)}, Val: {len(val_ds)}')

BS = 16
train_loader = DataLoader(train_ds, batch_size=BS, shuffle=True, collate_fn=collate, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=BS*2, shuffle=False, collate_fn=collate, num_workers=0)

# Model
class AudioClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = AutoModel.from_pretrained('FacebookAI/xlm-roberta-base')
        self.audio_proj = nn.Sequential(nn.Linear(768, 256), nn.ReLU(), nn.Dropout(0.2))
        self.gate = nn.Sequential(nn.Linear(1024, 256), nn.Sigmoid())
        self.classifier = nn.Sequential(nn.Linear(256, 64), nn.ReLU(), nn.Dropout(0.2), nn.Linear(64, 2))
    def forward(self, input_ids, attention_mask, audio_emb, phase='A'):
        txt = self.encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state[:,0,:]
        if phase == 'A':
            return self.classifier(txt)
        aud = self.audio_proj(audio_emb)
        g = self.gate(torch.cat([txt, aud], dim=-1))
        fused = g*txt + (1-g)*aud
        return self.classifier(fused)

model = AudioClassifier().to(DEVICE)
print(f'Params: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
#@title 7. Train Phase A
import time

# Phase A: freeze encoder, train rest
for p in model.encoder.parameters(): p.requires_grad = False
optimizer = AdamW([p for p in model.parameters() if p.requires_grad], lr=1e-3, weight_decay=0.01)
criterion = nn.CrossEntropyLoss()

best_f1 = 0
for ep in range(3):
    model.train()
    tloss, tpreds, tlabels = 0, [], []
    t0 = time.time()
    
    for bi, batch in enumerate(train_loader):
        logits = model(
            batch['input_ids'].to(DEVICE),
            batch['attention_mask'].to(DEVICE),
            batch['audio_emb'].to(DEVICE), phase='A')
        loss = criterion(logits, batch['labels'].to(DEVICE))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        optimizer.zero_grad()
        
        tloss += loss.item()
        tpreds.extend(logits.argmax(1).cpu().numpy())
        tlabels.extend(batch['labels'].numpy())
        
        if bi % 100 == 0:
            print(f'  Batch {bi}/{len(train_loader)}, loss={loss.item():.4f}', flush=True)
    
    tf1 = f1_score(tlabels, tpreds, zero_division=0)
    elapsed = time.time() - t0
    
    # Validate
    model.eval()
    vpreds, vlabels = [], []
    with torch.no_grad():
        for batch in val_loader:
            logits = model(batch['input_ids'].to(DEVICE), batch['attention_mask'].to(DEVICE), batch['audio_emb'].to(DEVICE), phase='A')
            vpreds.extend(logits.argmax(1).cpu().numpy())
            vlabels.extend(batch['labels'].numpy())
    
    vf1 = f1_score(vlabels, vpreds, zero_division=0)
    print(f'Epoch {ep+1}/3: train_f1={tf1:.4f}, val_f1={vf1:.4f}, time={elapsed:.0f}s', flush=True)
    
    if vf1 > best_f1:
        best_f1 = vf1
        torch.save(model.state_dict(), '/content/phaseA_best.pt')
        print(f'  ✅ Saved (f1={vf1:.4f})')
    
    gc.collect()
    torch.cuda.empty_cache()

print(f'\n✅ Phase A done! best_val_f1={best_f1:.4f}')